# Model 1: Pre vs Post Disaster Image Classification

**Goal:** Determine if a raw satellite image from the xBD dataset occurs before or after a disaster. We will use Transfer Learning (EfficientNetV2B0) instead of a custom shallow CNN to achieve significantly higher accuracy.

In [ ]:
import os
import glob
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2B0
import matplotlib.pyplot as plt

print('TensorFlow Version:', tf.__version__)

## 1. Data Preparation
Extract images and infer class based on the `_pre_` or `_post_` suffix in filenames.

In [ ]:
TRAIN_IMG_DIR = "../train/images"
TEST_IMG_DIR = "../test/images"

img_paths = glob.glob(os.path.join(TRAIN_IMG_DIR, "*.png"))
print(f"Total images found: {len(img_paths)}")

# Format: [ (path, label), ... ] where label 0=pre, 1=post
def parse_label(path):
    if "_pre_" in path: return 0
    return 1

labels = [parse_label(p) for p in img_paths]
# Note: In a real training pipeline, we use tf.data.Dataset.from_tensor_slices here


## 2. Model Architecture (Transfer Learning)
We replace the custom baseline CNN with EfficientNetV2B0 for immense hackathon performance.

In [ ]:
IMG_SIZE = (256, 256)

# Load pre-trained EfficientNet without the top classification layer
base_model = EfficientNetV2B0(
    include_top=False, 
    weights='imagenet', 
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False # Freeze base model initially

# Build the custom classification head
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    # Built-in Data Augmentation
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid') # Binary Output (Pre or Post)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()